# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule and its reason codes

I use a weighted baseline score to prioritize content items for human review.

The score combines two observable signals:

* **60% decline rate:** Content with a larger drop in impressions receives a higher score.
* **40% low kept impressions:** Content with fewer kept impressions receives a higher score relative to other items in the dataset.

The score is calculated as:

`opportunity_score = 0.6 × decline_rate + 0.4 × low_kept_component`

The decline rate is calculated by comparing impressions in the previous 30-day window with impressions in the latest 30-day window. Both components are normalized to a range of 0–1.

Reason codes explain why an item was selected:

* `strong_traffic_decline`
* `low_kept_impressions`
* `monitor`

The queue is sorted from highest to lowest opportunity score. This is a prioritization rule for review, not a prediction that refreshing content will definitely improve performance.


In [4]:
decline_weight = 0.60
low_kept_weight = 0.40

print("Baseline scoring weights")
print("Impression decline:", decline_weight)
print("Low kept impressions:", low_kept_weight)
print("Weights sum to:", decline_weight + low_kept_weight)

Baseline scoring weights
Impression decline: 0.6
Low kept impressions: 0.4
Weights sum to: 1.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Scoring Formula

For each content item, calculate the decline rate as:

`decline_rate = clip(1 - imp_last30 / imp_prev30, 0, 1)`

Then calculate the opportunity score:

`opportunity_score = 0.6 × decline_rate + 0.4 × (1 - kept_impressions / maximum_kept_impressions)`

The first component measures the observed reduction in impressions between the previous and latest 30-day windows. The second component gives a higher score to items with fewer kept impressions relative to the maximum in the analyzed table.

The previous-window impression threshold is 100. The score should be calculated only for rows that satisfy the data contract and have valid values. Zero denominators and missing values must be handled safely.


In [6]:
print("data exists:", "data" in globals())
print("features exists:", "features" in globals())
print("results exists:", "results" in globals())
print("ranked exists:", "ranked" in globals())
print("action_summary exists:", "action_summary" in globals())

data exists: False
features exists: False
results exists: False
ranked exists: False
action_summary exists: False


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

# Reuse the existing full-release results.
score_df = data.copy()

# Make sure the scoring fields exist.
required_columns = [
    "imp_prev30",
    "imp_last30",
    "kept_impressions",
]
missing = [c for c in required_columns if c not in score_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Calculate the two scoring components.
score_df["decline_rate"] = (
    1 - score_df["imp_last30"] /
    score_df["imp_prev30"].replace(0, np.nan)
).clip(0, 1)

max_kept = score_df["kept_impressions"].max()

if pd.isna(max_kept) or max_kept <= 0:
    score_df["low_kept_component"] = 0.0
else:
    score_df["low_kept_component"] = (
        1 - score_df["kept_impressions"] / max_kept
    ).clip(0, 1)

# Weighted opportunity score, on a 0–1 scale.
score_df["opportunity_score"] = (
    0.60 * score_df["decline_rate"]
    + 0.40 * score_df["low_kept_component"]
)

# Human-readable reason codes.
median_kept = score_df["kept_impressions"].median()

score_df["reason_code"] = np.select(
    [
        score_df["decline_rate"] >= 0.50,
        score_df["kept_impressions"] < median_kept,
    ],
    [
        "strong_traffic_decline",
        "low_kept_impressions",
    ],
    default="monitor",
)

# Recommended action.
score_df["action"] = np.select(
    [
        score_df["imp_last30"].fillna(0) == 0,
        score_df["decline_rate"] >= 0.50,
        score_df["decline_rate"] >= 0.20,
    ],
    [
        "investigate_traffic_loss",
        "review_or_refresh",
        "monitor_declining_performance",
    ],
    default="maintain_or_monitor",
)

# Rank every eligible item.
queue = (
    score_df[
        [
            "imp_prev30",
            "imp_last30",
            "kept_impressions",
            "decline_rate",
            "low_kept_component",
            "opportunity_score",
            "reason_code",
            "action",
        ]
    ]
    .dropna(subset=["opportunity_score"])
    .sort_values("opportunity_score", ascending=False)
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Save the CSV locally in the notebook runtime.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Rows ranked:", f"{len(queue):,}")
print("Output:", output_path)
display(queue.head(10))

NameError: name 'data' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

The top 20 volume-eligible items all have zero impressions in the latest window and a decline rate of 1.0. Their previous-window impressions range from 516 to 20,678, while kept impressions range from 10 to 15.

All 20 receive the reason code `strong_traffic_decline`. The opportunity scores range from approximately 0.999990 to 0.999993, with only six unique score values. This indicates that the score does not strongly distinguish items at the top of the queue.

**Recommended action:** Investigate traffic loss before deciding whether to refresh the content.

**Confidence:** The observed decline is clear in the supplied measurements, but confidence in the cause is limited. Missing data, tracking changes, seasonality, or differences between measurement windows could explain the pattern.

**What could make the recommendation wrong:** The apparent decline may reflect incomplete measurement rather than a genuine loss of traffic. The score is a review-prioritization heuristic, not proof that refreshing content will improve performance.


In [2]:
# Aggregate results recorded from the existing capstone analysis.
# This cell documents previously obtained results; it does not
# recalculate the warehouse or produce a new ranking.

existing_action_counts = {
    "Review or refresh content": 42756,
    "Maintain / monitor": 39700,
    "Monitor declining performance": 27631,
    "Investigate traffic loss": 1160,
}

print("Existing capstone action counts:")
for action, count in existing_action_counts.items():
    print(f"{action}: {count:,}")

print(
    "Total across recorded action categories:",
    f"{sum(existing_action_counts.values()):,}"
)

Existing capstone action counts:
Review or refresh content: 42,756
Maintain / monitor: 39,700
Monitor declining performance: 27,631
Investigate traffic loss: 1,160
Total across recorded action categories: 111,247


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks and Leakage Check

**Weak picks:** Items with zero recent impressions can receive the highest scores even when their previous-window impressions are relatively low. This can cause the queue to over-prioritize some items. A minimum threshold of 500 previous-window impressions was tested as a diagnostic safeguard; 84,180 items met this threshold.

**Leakage check:** The decline-rate score uses recent impressions to measure the decline being investigated. Therefore, this is a retrospective prioritization heuristic, not an independent prediction of future decline. The timing of `kept_impressions` must also be checked before claiming that all features are free from temporal leakage.

**Limitations:** Missing measurements, tracking changes, seasonality, and inconsistent window definitions may affect the ranking. The score does not establish why traffic declined or guarantee that refreshing content will improve performance.

**Decision:** Use the queue to guide human investigation, not as an automatic instruction to refresh content.


In [3]:
validation_summary = {
    "Random row split accuracy": 0.656,
    "Client-held-out accuracy": 0.611,
    "Majority baseline accuracy": 0.677,
    "Client-held-out with kept_impressions": 0.663,
}

print("Existing validation results:")
for name, value in validation_summary.items():
    print(f"{name}: {value:.3f}")

print(
    "Interpretation: client-held-out accuracy remains below "
    "the majority-class baseline."
)

Existing validation results:
Random row split accuracy: 0.656
Client-held-out accuracy: 0.611
Majority baseline accuracy: 0.677
Client-held-out with kept_impressions: 0.663
Interpretation: client-held-out accuracy remains below the majority-class baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.